### EDA
We will conduct EDA on the Customer Support ticket dataset as obtained from Kaggle

In [3]:
import pandas as pd
import numpy as np

In [2]:
import torch
from torch.utils.data import Dataset
from torch.utils.data import DataLoader

import torch.nn as nn
from transformers import AutoModel
from torch.optim import AdamW

In [3]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

In [4]:
df = pd.read_csv(r"/mnt/c/Rig/Pandoras BOX/DL/Datasets/customer_support_tickets.csv")

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df['Product Purchased'].nunique()

In [ ]:
print(df['Ticket Type'].unique(), '\n')
print(df['Ticket Subject'].unique())

In [5]:
df['Ticket Type'].value_counts()

Ticket Type
Refund request          1752
Technical issue         1747
Cancellation request    1695
Product inquiry         1641
Billing inquiry         1634
Name: count, dtype: int64

In [ ]:
df['Ticket Priority'].value_counts()

Data seems to be well distributed

In [ ]:
df.isna().sum()

In [5]:
df_drop = df.drop(columns = 
                  ['Ticket ID', 'Customer Name','Customer Email', 
                   'Customer Age', 'Customer Gender', 'Customer Satisfaction Rating'])

In [6]:
### IF NO IMPROVEMENT, REMOVE "PLEASE ASSIST" also then check ..... Now executing....Also try to change MAXLENGTH

In [10]:
for i in range(1,10):
    display(i,df_drop['Ticket Description'][i][:100])

1

"I'm having an issue with the {product_purchased}. Please assist.\n\nIf you need to change an existing "

2

"I'm facing a problem with my {product_purchased}. The {product_purchased} is not turning on. It was "

3

...

"I'm having an issue with the {product_purchased}. Please assist. (Thanks) I will contact all my supp"

8

"I'm having an issue with the {product_purchased}. Please assist. Thank you.\n\n{product_purchased} is "

9

'My {product_purchased} is making strange noises and not functioning properly. I suspect there might '

In [11]:
df_test = df_drop.copy()

In [13]:
df_test['New'] = df_test['Ticket Description'].str[65:].str.strip()

In [ ]:
# Alternatively......

df_test['New 2'] = df_test['Ticket Description'].str.split('.',n=1).str[1].str.strip()

In [14]:
df_test.head()

,Product Purchased,Date of Purchase,Ticket Type,Ticket Subject,Ticket Description,Ticket Status,Resolution,Ticket Priority,Ticket Channel,First Response Time,Time to Resolution,New
0,GoPro Hero,2021-03-22,Technical issue,Product setup,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Social media,2023-06-01 12:15:36,NaN,Your billing zip code is: 71701.\n\nWe appreci...
1,LG Smart TV,2021-05-22,Technical issue,Peripheral compatibility,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Chat,2023-06-01 16:45:38,NaN,If you need to change an existing product.\n\n...
2,Dell XPS,2020-07-14,Technical issue,Network problem,I'm facing a problem with my {product_purchase...,Closed,Case maybe show recently my computer follow.,Low,Social media,2023-06-01 11:14:38,2023-06-01 18:05:38,rchased} is not turning on. It was working fin...
3,Microsoft Office,2020-11-13,Billing inquiry,Account access,I'm having an issue with the {product_purchase...,Closed,Try capital clearly never color toward story.,Low,Social media,2023-06-01 07:29:40,2023-06-01 01:57:40,If you have a problem you're interested in and...
4,Autodesk AutoCAD,2020-02-04,Billing inquiry,Data loss,I'm having an issue with the {product_purchase...,Closed,West decision evidence bit.,Low,Email,2023-06-01 00:12:42,2023-06-01 19:53:42,Note: The seller is not responsible for any da...


In [ ]:
#del df_test

In [15]:
# df_drop["text"] = (
#     df["Ticket Subject"].astype(str) + " [SEP] " +
#     df["Ticket Description"].astype(str)
# )

df_drop["text"] = (
    df["Ticket Subject"].astype(str) + " [SEP] " +
    df_test["New"].astype(str)
)

In [16]:
df_drop['text']

0       Product setup [SEP] Your billing zip code is: ...
1       Peripheral compatibility [SEP] If you need to ...
2       Network problem [SEP] rchased} is not turning ...
3       Account access [SEP] If you have a problem you...
4       Data loss [SEP] Note: The seller is not respon...
                              ...                        
8464    Installation support [SEP] ng properly. I susp...
8465    Refund request [SEP] Sell for 30-35% (I also b...
8466    Account access [SEP] You are using a different...
8467    Payment issue [SEP] I don't think a product is...
8468    Hardware issue [SEP] The screen is flickering,...
Name: text, Length: 8469, dtype: object

In [17]:
df_drop.head()

,Product Purchased,Date of Purchase,Ticket Type,Ticket Subject,Ticket Description,Ticket Status,Resolution,Ticket Priority,Ticket Channel,First Response Time,Time to Resolution,text
0,GoPro Hero,2021-03-22,Technical issue,Product setup,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Social media,2023-06-01 12:15:36,NaN,Product setup [SEP] Your billing zip code is: ...
1,LG Smart TV,2021-05-22,Technical issue,Peripheral compatibility,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Chat,2023-06-01 16:45:38,NaN,Peripheral compatibility [SEP] If you need to ...
2,Dell XPS,2020-07-14,Technical issue,Network problem,I'm facing a problem with my {product_purchase...,Closed,Case maybe show recently my computer follow.,Low,Social media,2023-06-01 11:14:38,2023-06-01 18:05:38,Network problem [SEP] rchased} is not turning ...
3,Microsoft Office,2020-11-13,Billing inquiry,Account access,I'm having an issue with the {product_purchase...,Closed,Try capital clearly never color toward story.,Low,Social media,2023-06-01 07:29:40,2023-06-01 01:57:40,Account access [SEP] If you have a problem you...
4,Autodesk AutoCAD,2020-02-04,Billing inquiry,Data loss,I'm having an issue with the {product_purchase...,Closed,West decision evidence bit.,Low,Email,2023-06-01 00:12:42,2023-06-01 19:53:42,Data loss [SEP] Note: The seller is not respon...


In [18]:
df_drop[["Ticket Type", "Ticket Priority", "text"]].sample(5)

,Ticket Type,Ticket Priority,text
7130,Product inquiry,High,Software bug [SEP] Thanks!\n\n\nI'm having an ...
7598,Refund request,Low,Installation support [SEP] How do I contact hi...
3378,Product inquiry,High,Product compatibility [SEP] 1411\n\n10.06.2016...
3237,Product inquiry,Low,Hardware issue [SEP] Thanks. I am also having ...
4189,Technical issue,High,Display issue [SEP] ed}. Is there any way to r...


In [19]:
df_drop['text'][0]

"Product setup [SEP] Your billing zip code is: 71701.\n\nWe appreciate that you have requested a website address.\n\nPlease double check your email address. I've tried troubleshooting steps mentioned in the user manual, but the issue persists."

In [20]:
df_drop.head(1)

,Product Purchased,Date of Purchase,Ticket Type,Ticket Subject,Ticket Description,Ticket Status,Resolution,Ticket Priority,Ticket Channel,First Response Time,Time to Resolution,text
0,GoPro Hero,2021-03-22,Technical issue,Product setup,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Social media,2023-06-01 12:15:36,NaN,Product setup [SEP] Your billing zip code is: ...


In [21]:
df_new = df_drop[['Ticket Type', 'Ticket Subject', 'Ticket Description','Ticket Priority', 'text']]

## Banking77 Dataset will be used
This dataset is pretty useless, and yielding  ~20 % accuracy. Lets try it out

In [ ]:
df_new.head()

### Data Preprocessing
We will conduct preprocessing, encoding, and splitting of data

In [22]:
df1 = df_new.copy()

In [23]:
df1['text'][0]

"Product setup [SEP] Your billing zip code is: 71701.\n\nWe appreciate that you have requested a website address.\n\nPlease double check your email address. I've tried troubleshooting steps mentioned in the user manual, but the issue persists."

In [27]:
for i in range(1,15):
    display(i,df1['text'][i][:250])

1

"Peripheral compatibility [SEP] If you need to change an existing product.\n\nI'm having an issue with the {product_purchased}. Please assist.\n\nIf The issue I'm facing is intermittent. Sometimes it works fine, but other times it acts up unexpectedly."

2

"Network problem [SEP] rchased} is not turning on. It was working fine until yesterday, but now it doesn't respond.\n\n1.8.3 I really I'm using the original charger that came with my {product_purchased}, but it's not charging properly."

3

...

"Hardware issue [SEP] CQW: Why didn't I send him the invoice? Thanks a lot.\n\nL: He's like the best customer I've met. I've noticed that the issue occurs consistently when I use a specific feature or application on my {product_purchased}."

13

"Product setup [SEP] ect to any available networks. What steps should I take to troubleshoot this issue?\n\nI can't find the 'Product_IP' field of the I'm concerned about the security of my {product_purchased} and would like to ensure that my data is sa"

14

"Product setup [SEP] Product Name: TPUBASK3E3KQ0\n\n\nJoin Date: Oct 2007 Posts: 11,532\n\nQuote: I've recently updated the firmware of my {product_purchased}, and the issue started happening afterward. Could it be related to the update?"

In [28]:
intent_encoder = LabelEncoder()
urgency_encoder = LabelEncoder()

df1.loc[:,'intent_label'] = intent_encoder.fit_transform(df['Ticket Type']) 
df1.loc[:,'urgency_label'] = urgency_encoder.fit_transform(df['Ticket Priority'])

In [29]:
df1.head(5)

,Ticket Type,Ticket Subject,Ticket Description,Ticket Priority,text,intent_label,urgency_label
0,Technical issue,Product setup,I'm having an issue with the {product_purchase...,Critical,Product setup [SEP] Your billing zip code is: ...,4,0
1,Technical issue,Peripheral compatibility,I'm having an issue with the {product_purchase...,Critical,Peripheral compatibility [SEP] If you need to ...,4,0
2,Technical issue,Network problem,I'm facing a problem with my {product_purchase...,Low,Network problem [SEP] rchased} is not turning ...,4,2
3,Billing inquiry,Account access,I'm having an issue with the {product_purchase...,Low,Account access [SEP] If you have a problem you...,0,2
4,Billing inquiry,Data loss,I'm having an issue with the {product_purchase...,Low,Data loss [SEP] Note: The seller is not respon...,0,2


In [30]:
# from sklearn.model_selection import train_test_split

# X = df1["text"] 
# y_intent = df1["intent_label"] 
# y_urgency = df1["urgency_label"] 

# X_train, X_test, y_intent_train, y_intent_test, y_urgency_train, y_urgency_test = train_test_split(
#     X,
#     y_intent,
#     y_urgency,
#     test_size=0.2,
#     random_state=42,
#     stratify=y_intent
# )

# len(X_train), len(X_test)

In [31]:
# ---------------FOR SINGLE TARGET PREDICTION-----------------
from sklearn.model_selection import train_test_split

X = df1["text"] 
y_intent = df1["intent_label"] 

X_train, X_test, y_intent_train, y_intent_test = train_test_split(
    X,
    y_intent,
    
    test_size=0.2,
    random_state=42,
    stratify=y_intent
)

len(X_train), len(X_test)

(6775, 1694)

In [32]:
# During training, key error was popping up.
# after traintestsplit, pandas keeps the original indices, and may be like: [12, 45, 78, 102, 2950, ...]
# later, on using self.intent_label[idx], idx is 1,2,3...., if 2950 is not found, KeyError is thrown



X_train = X_train.reset_index(drop=True)
X_test = X_test.reset_index(drop=True)

y_intent_train = y_intent_train.reset_index(drop=True)
y_intent_test = y_intent_test.reset_index(drop=True)

# y_urgency_train = y_urgency_train.reset_index(drop=True)
# y_urgency_test = y_urgency_test.reset_index(drop=True)


### Tokenization
We invoke Transformers here on out, for tokenization of our text(Ticket Description)

In [33]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

In [34]:
# Tokenizing

train_encodings = tokenizer(
    X_train.to_list(),
    truncation=True,
    padding=True,
    max_length=250
)

test_encodings = tokenizer(
    X_test.tolist(),
    truncation=True,
    padding=True,
    max_length=250
)

In [ ]:
# # Creating a Custom PyTorch Dataset
# # Dataset is something that i) knows its size  & ii) can return a value at a time

# class TicketDataset(Dataset):
#     def __init__(self, encodings, intent_labels, urgency_labels):
#         self.encodings = encodings
#         self.intent_labels = intent_labels
#         self.urgency_labels = urgency_labels

#     def __len__(self):
#         return len(self.intent_labels)

#     def __getitem__(self, idx):
#         item = {
#             "input_ids": torch.tensor(self.encodings["input_ids"][idx]),
#             "attention_mask": torch.tensor(self.encodings["attention_mask"][idx]),
#             "intent_labels": torch.tensor(self.intent_labels[idx]),
#             "urgency_labels": torch.tensor(self.urgency_labels[idx])
#         }
#         return item


In [35]:
# -------------------- FOR SINGLE TARGET PREDICTION------------------------ 

class TicketDataset(Dataset):
    def __init__(self, encodings, intent_labels):
        self.encodings = encodings
        self.intent_labels = intent_labels
        #self.urgency_labels = urgency_labels

    def __len__(self):
        return len(self.intent_labels)

    def __getitem__(self, idx):
        item = {
            "input_ids": torch.tensor(self.encodings["input_ids"][idx]),
            "attention_mask": torch.tensor(self.encodings["attention_mask"][idx]),
            "intent_labels": torch.tensor(self.intent_labels[idx]),
            #"urgency_labels": torch.tensor(self.urgency_labels[idx])
        }
        return item


In [ ]:
# Alternate way: Create an HF Dataset.
# This is a trade-off, we have lesser control, and its more difficult to customize later

# from datasets import Dataset

# dataset = Dataset.from_dict({
#     "input_ids": train_encodings["input_ids"],
#     "attention_mask": train_encodings["attention_mask"],
#     "intent_labels": y_intent_train,
#     "urgency_labels": y_urgency_train
# })

In [36]:
train_dataset = TicketDataset(
    train_encodings,
    y_intent_train,
    #y_urgency_train
)

test_dataset = TicketDataset(
    test_encodings,
    y_intent_test,
    #y_urgency_test
)

In [37]:
# Data Loader

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False
)

# Each batch is formed of 4 keys(as above) and is a tensor of value: (batch_size, sequence_length)

Work flow:
Text --> BERT Encoder --> [CLS token]Embedding  ->  Intent head + Urgency head

In [ ]:
# class MultiTaskBert(nn.Module):
    
#     def __init__(self, num_intent_classes, num_urgency_classes):
#         super().__init__()
        
#         self.bert = AutoModel.from_pretrained("bert-base-uncased")
#         hidden_state = self.bert.config.hidden_size
        
#         self.intent_classifier = nn.Linear(hidden_state, num_intent_classes)   #Taking in vec of dim hiddenstate and output of dim: no. of classes
#         self.urgency_classifier = nn.Linear(hidden_state, num_urgency_classes)

        
#     def forward(self, input_ids, attention_mask):
#         outputs = self.bert(
#             input_ids = input_ids,
#             attention_mask = attention_mask #(16,128)
#         )                                                                      # dim is (16,128,768)

#         cls_embeddings = outputs.last_hidden_state[:, 0, :]                  # Extracting CLS token, which acts as summary repr. of the sentence
        
#         intent_logits =  self.intent_classifier(cls_embeddings)
#         urgency_logits =  self.urgency_classifier(cls_embeddings)
#         return intent_logits, urgency_logits

In [38]:
# -----------------------FOR SINGLE TARGET PREIDCTION-------------------

class MultiTaskBert(nn.Module):
    
    def __init__(self, num_intent_classes):
        super().__init__()
        
        self.bert = AutoModel.from_pretrained("bert-base-uncased")
        hidden_state = self.bert.config.hidden_size
        self.intent_classifier = nn.Linear(hidden_state, num_intent_classes)   #Taking in vec of dim hiddenstate and output of dim: no. of classes
        
    def forward(self, input_ids, attention_mask):
        outputs = self.bert(
            input_ids = input_ids,
            attention_mask = attention_mask #(16,128)
        )                                                                      # dim is (16,128,768)

        cls_embeddings = outputs.last_hidden_state[:, 0, :]                  # Extracting CLS token, which acts as summary repr. of the sentence
        intent_logits =  self.intent_classifier(cls_embeddings)
        return intent_logits

Loss and Optim claculation(Will prroceed to Training)

In [39]:
intent_loss_fn = nn.CrossEntropyLoss()
#urgency_loss_fn = nn.CrossEntropyLoss()

In [40]:
num_intent_classes = df1['intent_label'].nunique()
#num_urgency_classes = df1['urgency_label'].nunique()

#model = MultiTaskBert(num_intent_classes, num_urgency_classes)
model = MultiTaskBert(num_intent_classes)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [41]:
optimizer = AdamW(model.parameters(), lr=2e-5)

In [42]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device, '\n')
model.to(device)

cuda 



MultiTaskBert(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_aff

In [ ]:
# # -----------------Training-----------------
# epochs = 10

# for epoch in range(epochs):
#     model.train()
    
#     total_loss = 0
    
#     for batch in train_loader:
#         optimizer.zero_grad()
        
#         input_ids = batch["input_ids"].to(device)
#         attention_mask = batch["attention_mask"].to(device)
#         intent_labels = batch["intent_labels"].to(device)
#         urgency_labels = batch["urgency_labels"].to(device)
        
#         intent_logits, urgency_logits = model(input_ids, attention_mask)
        
#         intent_loss = intent_loss_fn(intent_logits, intent_labels)
#         urgency_loss = urgency_loss_fn(urgency_logits, urgency_labels)
        
#         loss = (intent_loss + urgency_loss)/2
#         loss.backward()
        
#         optimizer.step()
#         total_loss += loss.item()
    
#     avg_loss = total_loss / len(train_loader)

#     print(f"\nEpoch: {epoch+1}/{epochs}")
#     print("Training loss:", avg_loss)

In [ ]:
# # ------------------VALIDATION-----------------
# import time
# start = time.time()

# epochs = 10

# for epoch in range(epochs):
 
#     model.eval()

#     intent_preds = []
#     intent_true = []
#     urgency_preds = []
#     urgency_true = []

#     with torch.no_grad():
#         for batch in test_loader:

#             input_ids = batch["input_ids"].to(device)
#             attention_mask = batch["attention_mask"].to(device)
#             intent_labels = batch["intent_labels"].to(device)
#             urgency_labels = batch["urgency_labels"].to(device)

#             intent_logits, urgency_logits = model(input_ids, attention_mask)

#             intent_pred = torch.argmax(intent_logits, dim=1)
#             urgency_pred = torch.argmax(urgency_logits, dim=1)

#             intent_preds.extend(intent_pred.cpu().numpy())
#             intent_true.extend(intent_labels.cpu().numpy())

#             urgency_preds.extend(urgency_pred.cpu().numpy())
#             urgency_true.extend(urgency_labels.cpu().numpy())


#     print(f"\nEpoch {epoch+1}/{epochs}")
#     print("Intent Acc:", round(accuracy_score(intent_true, intent_preds),4))
#     print("Intent F1:",f1_score(intent_true, intent_preds, average="weighted"))
#     print("Urgency Acc:", round(accuracy_score(urgency_true, urgency_preds),4)) 
#     print("Urgency F1:", f1_score(urgency_true, urgency_preds, average="weighted"))
# print(f"Time Taken = {time.time() - start}")

In [ ]:
# #combined loop for training and validation:

# epochs = 10

# for epoch in range(epochs):

#     # ----------------- TRAINING -----------------
#     model.train()
#     total_loss = 0

#     for batch in train_loader:
#         optimizer.zero_grad()

#         input_ids = batch["input_ids"].to(device)
#         attention_mask = batch["attention_mask"].to(device)
#         intent_labels = batch["intent_labels"].to(device)
#         urgency_labels = batch["urgency_labels"].to(device)

#         intent_logits, urgency_logits = model(input_ids, attention_mask)

#         intent_loss = intent_loss_fn(intent_logits, intent_labels)
#         urgency_loss = urgency_loss_fn(urgency_logits, urgency_labels)

#         loss = (intent_loss + urgency_loss) / 2
#         loss.backward()
#         optimizer.step()

#         total_loss += loss.item()

#     avg_loss = total_loss / len(train_loader)

#     # ----------------- VALIDATION -----------------
#     model.eval()

#     intent_preds = []
#     intent_true = []
#     urgency_preds = []
#     urgency_true = []

#     with torch.no_grad():
#         for batch in test_loader:   # preferably val_loader

#             input_ids = batch["input_ids"].to(device)
#             attention_mask = batch["attention_mask"].to(device)
#             intent_labels = batch["intent_labels"].to(device)
#             urgency_labels = batch["urgency_labels"].to(device)

#             intent_logits, urgency_logits = model(input_ids, attention_mask)

#             intent_pred = torch.argmax(intent_logits, dim=1)
#             urgency_pred = torch.argmax(urgency_logits, dim=1)

#             intent_preds.extend(intent_pred.cpu().numpy())
#             intent_true.extend(intent_labels.cpu().numpy())

#             urgency_preds.extend(urgency_pred.cpu().numpy())
#             urgency_true.extend(urgency_labels.cpu().numpy())

#     # ----------------- METRICS -----------------
#     intent_acc = accuracy_score(intent_true, intent_preds)
#     intent_f1 = f1_score(intent_true, intent_preds, average="weighted")

#     urgency_acc = accuracy_score(urgency_true, urgency_preds)
#     urgency_f1 = f1_score(urgency_true, urgency_preds, average="weighted")

#     print(f"\nEpoch {epoch+1}/{epochs}")
#     print("Training Loss:", round(avg_loss, 4))
#     print("Intent Acc:", round(intent_acc, 4))
#     print("Intent F1:", round(intent_f1, 4))
#     print("Urgency Acc:", round(urgency_acc, 4))
#     print("Urgency F1:", round(urgency_f1, 4))


In [43]:
# ---------------------------FOR SINGLE TRAGET PREDICTION---------------------

epochs = 10
for epoch in range(epochs):

    # ----------------- TRAINING -----------------
    model.train()
    total_loss = 0

    for batch in train_loader:
        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        intent_labels = batch["intent_labels"].to(device)
 
        intent_logits = model(input_ids, attention_mask)

        intent_loss = intent_loss_fn(intent_logits, intent_labels)

        loss = intent_loss
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    # ----------------- VALIDATION -----------------
    model.eval()

    intent_preds = []
    intent_true = []

    with torch.no_grad():
        for batch in test_loader:   

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            intent_labels = batch["intent_labels"].to(device)
 
            intent_logits = model(input_ids, attention_mask)

            intent_pred = torch.argmax(intent_logits, dim=1)
 
            intent_preds.extend(intent_pred.cpu().numpy())
            intent_true.extend(intent_labels.cpu().numpy())
 
    # ----------------- METRICS -----------------
    intent_acc = accuracy_score(intent_true, intent_preds)
    intent_f1 = f1_score(intent_true, intent_preds, average="weighted")
 
    print(f"\nEpoch {epoch+1}/{epochs}")
    print("Training Loss:", round(avg_loss, 4))
    print("Intent Acc:", round(intent_acc, 4))
    print("Intent F1:", round(intent_f1, 4))



Epoch 1/10
Training Loss: 1.6172
Intent Acc: 0.2084
Intent F1: 0.0846

Epoch 2/10
Training Loss: 1.6147
Intent Acc: 0.2054
Intent F1: 0.1652

Epoch 3/10
Training Loss: 1.6103
Intent Acc: 0.2131
Intent F1: 0.1124

Epoch 4/10
Training Loss: 1.6005
Intent Acc: 0.193
Intent F1: 0.1771

Epoch 5/10
Training Loss: 1.5378
Intent Acc: 0.2019
Intent F1: 0.1713

Epoch 6/10
Training Loss: 1.263
Intent Acc: 0.1972
Intent F1: 0.1949

Epoch 7/10
Training Loss: 0.8035
Intent Acc: 0.1936
Intent F1: 0.1888

Epoch 8/10
Training Loss: 0.4415
Intent Acc: 0.2031
Intent F1: 0.1987

Epoch 9/10
Training Loss: 0.287
Intent Acc: 0.209
Intent F1: 0.2064

Epoch 10/10
Training Loss: 0.21
Intent Acc: 0.2096
Intent F1: 0.2081
